# Data Quality Assessment

## Business Objective

The objective of this notebook is to assess the quality and reliability of the raw e-commerce datasets before any cleaning, integration, or analysis is performed.

This assessment evaluates the following six tables:

- `users`
- `products`
- `orders`
- `order_items`
- `reviews`
- `events`

The purpose is to identify data quality issues such as missing values, duplicate records, invalid values, inconsistent formatting, incorrect datatypes, and primary key violations. No cleaning operations are performed in this notebook. All identified issues will be addressed later in the Data Cleaning phase.

## Methodology

Each dataset is assessed using a consistent framework:

1. Dataset structure  
2. Column overview  
3. Datatype inspection  
4. Missing value assessment  
5. Duplicate row assessment  
6. Primary key uniqueness check  
7. Business-rule validation  
8. Summary of key findings  

This structure ensures that each table is evaluated consistently and that cleaning decisions are based on documented evidence rather than assumptions.

## Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

## Set Project Directory

In [ ]:
os.chdir("/Users/saadmaher/Desktop/Data Science/Project portfolio/ecommerce-data-analysis")

PROJECT_ROOT = Path(os.getcwd())
DATA_DIR = PROJECT_ROOT / "data" / "raw"

PROJECT_ROOT

## Verify Available Raw Datasets

In [ ]:
list(DATA_DIR.iterdir())

## Load Raw Datasets

In [ ]:
users = pd.read_csv(DATA_DIR / "users_raw.csv")
products = pd.read_csv(DATA_DIR / "products_raw.csv")
orders = pd.read_csv(DATA_DIR / "orders_raw.csv")
order_items = pd.read_csv(DATA_DIR / "order_items_raw.csv")
reviews = pd.read_csv(DATA_DIR / "reviews_raw.csv")
events = pd.read_csv(DATA_DIR / "events_raw.csv")

## Helper Functions

To avoid repeating the same code for every table, reusable helper functions are created below. These functions summarize structure, missing values, duplicate records, and primary key uniqueness for each dataset.

In [ ]:
def dataset_overview(df, table_name):
    print(f"===== {table_name.upper()} DATASET OVERVIEW =====")
    print(f"Rows: {df.shape[0]:,}")
    print(f"Columns: {df.shape[1]:,}")
    print("\nColumn Names:")
    print(list(df.columns))
    print("\nData Types:")
    print(df.dtypes)


def missing_values_report(df):
    missing_count = df.isna().sum()
    missing_percent = (df.isna().mean() * 100).round(2)

    report = pd.DataFrame({
        "missing_count": missing_count,
        "missing_percent": missing_percent
    })

    return report[report["missing_count"] > 0].sort_values(
        by="missing_percent",
        ascending=False
    )


def duplicate_rows_report(df):
    duplicate_count = df.duplicated().sum()
    duplicate_percent = round(df.duplicated().mean() * 100, 2)

    return pd.DataFrame({
        "duplicate_rows": [duplicate_count],
        "duplicate_percent": [duplicate_percent]
    })


def primary_key_report(df, key_column):
    total_rows = len(df)
    unique_keys = df[key_column].nunique(dropna=False)
    duplicated_keys = df[key_column].duplicated().sum()
    missing_keys = df[key_column].isna().sum()

    return pd.DataFrame({
        "key_column": [key_column],
        "total_rows": [total_rows],
        "unique_keys": [unique_keys],
        "duplicated_keys": [duplicated_keys],
        "missing_keys": [missing_keys]
    })

# 1. Users Data Quality Assessment

## Business Context

The Users table represents customer profile information. Each row should represent one unique customer. This table is important because it connects customer behavior, orders, reviews, and events through `user_id`.

## Expected Data Quality Rules

- `user_id` should uniquely identify each customer.
- `user_id` should not be missing.
- Exact duplicate records should not exist.
- `signup_date` should represent a valid date.
- Demographic fields such as `gender` and `city` should be checked for completeness and consistency.

In [ ]:
dataset_overview(users, "users")

## Users Missing Values

In [ ]:
missing_values_report(users)

## Users Duplicate Rows

In [ ]:
duplicate_rows_report(users)

## Users Primary Key Check

In [ ]:
primary_key_report(users, "user_id")

## Users Invalid Date Check

In [ ]:
invalid_signup_dates = users[
    pd.to_datetime(users["signup_date"], errors="coerce").isna()
]

invalid_signup_dates.shape[0]

## Users Interpretation

The Users table should contain one record per customer. The assessment checks whether the customer identifier is unique, whether profile information is complete, whether duplicate rows exist, and whether the `signup_date` field can be converted into a valid datetime format.

Any duplicate customer identifiers, missing profile fields, or invalid signup dates should be addressed during the Data Cleaning phase.

# 2. Products Data Quality Assessment

## Business Context

The Products table contains the product catalog. This table is central to sales, category, revenue, and product performance analysis.

## Expected Data Quality Rules

- `product_id` should uniquely identify each product.
- `product_id` should not be missing.
- Exact duplicate product records should not exist.
- `price` should be positive.
- `rating` should be within a valid range, typically 1 to 5.
- Product categories should be consistent.

In [ ]:
dataset_overview(products, "products")

## Products Missing Values

In [ ]:
missing_values_report(products)

## Products Duplicate Rows

In [ ]:
duplicate_rows_report(products)

## Products Primary Key Check

In [ ]:
primary_key_report(products, "product_id")

## Products Price Validity Check

In [ ]:
invalid_prices = products[products["price"] <= 0]
invalid_prices.shape[0]

## Products Rating Validity Check

In [ ]:
invalid_product_ratings = products[
    (products["rating"] < 1) | (products["rating"] > 5)
]

invalid_product_ratings.shape[0]

## Products Category Consistency Check

In [ ]:
products["category"].value_counts(dropna=False)

## Products Interpretation

The Products table is expected to provide a reliable product catalog for revenue and category-level analysis. Key risks include duplicate product identifiers, missing product information, invalid prices, ratings outside the accepted range, and inconsistent category labels.

These issues may affect product-level reporting and business intelligence dashboards if not corrected during cleaning.

# 3. Orders Data Quality Assessment

## Business Context

The Orders table contains order-level transactions. It is essential for analyzing revenue, order volume, customer purchasing behavior, and business growth over time.

## Expected Data Quality Rules

- `order_id` should uniquely identify each order.
- `order_id` should not be missing.
- `user_id` should link each order to a valid customer.
- `order_date` should be a valid datetime value.
- `order_status` should contain valid business statuses.
- `total_amount` should not be negative.

In [ ]:
dataset_overview(orders, "orders")

## Orders Missing Values

In [ ]:
missing_values_report(orders)

## Orders Duplicate Rows

In [ ]:
duplicate_rows_report(orders)

## Orders Primary Key Check

In [ ]:
primary_key_report(orders, "order_id")

## Orders Invalid Date Check

In [ ]:
invalid_order_dates = orders[
    pd.to_datetime(orders["order_date"], errors="coerce").isna()
]

invalid_order_dates.shape[0]

## Orders Total Amount Validity Check

In [ ]:
invalid_order_amounts = orders[orders["total_amount"] < 0]
invalid_order_amounts.shape[0]

## Orders Status Consistency Check

In [ ]:
orders["order_status"].value_counts(dropna=False)

## Orders Interpretation

The Orders table is one of the most important transactional tables in the database. Data quality issues in this table can directly affect revenue reporting, sales trends, order status analysis, and customer purchasing insights.

Invalid dates, negative order amounts, missing statuses, duplicated order identifiers, or inconsistent order status labels should be investigated and cleaned before analysis.

# 4. Order Items Data Quality Assessment

## Business Context

The Order Items table contains item-level purchase details. It connects orders to products and enables detailed analysis of product sales, quantities, basket composition, and revenue contribution.

## Expected Data Quality Rules

- `order_item_id` should uniquely identify each order item.
- `order_id` should link each item to a valid order.
- `product_id` should link each item to a valid product.
- `quantity` should be greater than zero.
- `item_price` should be positive.
- Exact duplicate rows should not exist.

In [ ]:
dataset_overview(order_items, "order_items")

## Order Items Missing Values

In [ ]:
missing_values_report(order_items)

## Order Items Duplicate Rows

In [ ]:
duplicate_rows_report(order_items)

## Order Items Primary Key Check

In [ ]:
primary_key_report(order_items, "order_item_id")

## Quantity Validity Check

In [ ]:
invalid_quantities = order_items[order_items["quantity"] <= 0]
invalid_quantities.shape[0]

## Item Price Validity Check

In [ ]:
invalid_item_prices = order_items[order_items["item_price"] <= 0]
invalid_item_prices.shape[0]

## Order Items Interpretation

The Order Items table determines product-level sales and revenue calculations. Invalid quantities or item prices may distort revenue and product performance metrics. Referential integrity with Orders and Products should also be validated during the Data Integration phase.

# 5. Reviews Data Quality Assessment

## Business Context

The Reviews table contains customer product feedback. It supports customer satisfaction analysis, product quality evaluation, and rating-based insights.

## Expected Data Quality Rules

- `review_id` should uniquely identify each review.
- `user_id` should link each review to a valid customer.
- `product_id` should link each review to a valid product.
- `rating` should be within a valid range, typically 1 to 5.
- `review_date` should be a valid date.
- Review text may be missing, but missing values should be documented.

In [ ]:
dataset_overview(reviews, "reviews")

## Reviews Missing Values

In [ ]:
missing_values_report(reviews)

## Reviews Duplicate Rows

In [ ]:
duplicate_rows_report(reviews)

## Reviews Primary Key Check

In [ ]:
primary_key_report(reviews, "review_id")

## Review Rating Validity Check

In [ ]:
invalid_review_ratings = reviews[
    (reviews["rating"] < 1) | (reviews["rating"] > 5)
]

invalid_review_ratings.shape[0]

## Review Date Validity Check

In [ ]:
invalid_review_dates = reviews[
    pd.to_datetime(reviews["review_date"], errors="coerce").isna()
]

invalid_review_dates.shape[0]

## Reviews Interpretation

The Reviews table is important for understanding customer satisfaction and product perception. Invalid ratings, invalid review dates, duplicate review identifiers, or missing review text should be documented and handled appropriately during cleaning.

# 6. Events Data Quality Assessment

## Business Context

The Events table captures user behavior such as product views, cart additions, wishlist actions, and purchases. It supports funnel analysis, conversion tracking, and behavioral analytics.

## Expected Data Quality Rules

- `event_id` should uniquely identify each event.
- `user_id` should link each event to a valid user.
- `product_id` should link each event to a valid product when applicable.
- `event_type` should contain valid event categories.
- `event_timestamp` should be a valid datetime value.
- Exact duplicate event records should not exist.

In [ ]:
dataset_overview(events, "events")

## Events Missing Values

In [ ]:
missing_values_report(events)

## Events Duplicate Rows

In [ ]:
duplicate_rows_report(events)

## Events Primary Key Check

In [ ]:
primary_key_report(events, "event_id")

## Event Type Consistency Check

In [ ]:
events["event_type"].value_counts(dropna=False)

## Event Timestamp Validity Check

In [ ]:
invalid_event_timestamps = events[
    pd.to_datetime(events["event_timestamp"], errors="coerce").isna()
]

invalid_event_timestamps.shape[0]

## Events Interpretation

The Events table is critical for behavioral and funnel analysis. Invalid timestamps, missing event types, duplicate events, or inconsistent event labels may affect conversion rate calculations and customer journey analysis.

# Overall Data Quality Report

The data quality assessment identified issues across multiple tables, including potential missing values, duplicate records, invalid dates, invalid numeric values, inconsistent categorical formatting, and primary key violations.

These findings confirm that the raw datasets require a structured cleaning phase before they can be safely used for analysis, reporting, dashboard development, or business decision-making.

## Data Quality Summary Table

In [ ]:
quality_summary = pd.DataFrame({
    "table": [
        "users",
        "products",
        "orders",
        "order_items",
        "reviews",
        "events"
    ],
    "rows": [
        len(users),
        len(products),
        len(orders),
        len(order_items),
        len(reviews),
        len(events)
    ],
    "columns": [
        users.shape[1],
        products.shape[1],
        orders.shape[1],
        order_items.shape[1],
        reviews.shape[1],
        events.shape[1]
    ],
    "duplicate_rows": [
        users.duplicated().sum(),
        products.duplicated().sum(),
        orders.duplicated().sum(),
        order_items.duplicated().sum(),
        reviews.duplicated().sum(),
        events.duplicated().sum()
    ],
    "missing_values_total": [
        users.isna().sum().sum(),
        products.isna().sum().sum(),
        orders.isna().sum().sum(),
        order_items.isna().sum().sum(),
        reviews.isna().sum().sum(),
        events.isna().sum().sum()
    ]
})

quality_summary

# Executive Summary

## Objective

The objective of this notebook was to assess the quality of the raw e-commerce database before cleaning, integration, and analysis. Six datasets were reviewed: Users, Products, Orders, Order Items, Reviews, and Events.

## Key Assessment Areas

The assessment focused on:

- Dataset structure
- Missing values
- Duplicate records
- Primary key uniqueness
- Invalid dates
- Invalid numeric values
- Categorical consistency
- Business-rule violations

## Main Findings

The assessment confirmed that the raw datasets contain several data quality issues that must be addressed before analysis. These include missing values, duplicate rows, duplicate identifiers, invalid date values, invalid numeric values, and inconsistent categorical formatting.

## Recommendation

Before performing exploratory analysis, SQL analysis, or dashboard development, the identified issues should be handled in a dedicated Data Cleaning notebook. Cleaning should be performed systematically, with every transformation documented and validated.

## Next Steps

The next stage of the project is the Data Cleaning phase, where each table will be cleaned according to the issues identified in this assessment. After cleaning, the datasets will be validated and integrated to support downstream business analysis.

## Project Methodology

This project follows a structured analytical workflow:

**Raw Data**  
↓  
**Data Quality Assessment**  
↓  
**Data Cleaning**  
↓  
**Validation**  
↓  
**Data Integration**  
↓  
**Exploratory Data Analysis**  
↓  
**SQL Business Analysis**  
↓  
**Dashboard Development**  
↓  
**Executive Reporting**

This workflow ensures that all insights are built on reliable, transparent, and well-documented data.